# 02 – Chunking Lirik
Memecah tiap lagu menjadi jendela baris berurutan dengan tumpang tindih. Tiap chunk menyimpan `song_id`, posisi baris,
teks tampilan (`text`), dan teks ternormalisasi untuk indeks leksikal (`text_lex`).
Kolom `n_songs_same_text` dipakai notebook 05 untuk membuang kueri ambigu.

In [1]:
import sys, os, glob
import numpy as np, pandas as pd

path = glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-clean/songs_clean.parquet", recursive=True)[0]
sys.path.insert(0, os.path.dirname(path))
from lyrecall_text import normalize_lex, make_chunks

CHUNK_LINES = 3     # ablasi: 2, 3, 4
STRIDE      = 2     # tumpang tindih = CHUNK_LINES - STRIDE baris
OUT_DIR     = "/kaggle/working"

df = pd.read_parquet(path)
print("Lagu:", len(df))

Lagu: 99821


In [2]:
sids, starts, ends, texts = [], [], [], []
for sid, lyr in zip(df["song_id"].values, df["lyrics"].values):
    lines = lyr.split("\n")
    for a, b in make_chunks(len(lines), CHUNK_LINES, STRIDE):
        sids.append(sid); starts.append(a); ends.append(b)
        texts.append("\n".join(lines[a:b]))

chunks = pd.DataFrame({"song_id": sids, "start_line": starts, "end_line": ends, "text": texts})
del sids, starts, ends, texts
print("Chunk mentah:", len(chunks))

Chunk mentah: 2056641


In [3]:
# normalisasi, buang duplikat, tandai ambigu
chunks["text_lex"] = chunks["text"].apply(normalize_lex)
chunks = chunks[chunks["text_lex"].str.len() > 0]

n0 = len(chunks)
chunks = chunks.drop_duplicates(["song_id", "text_lex"]).reset_index(drop=True)   # refren berulang dalam satu lagu
print(f"Duplikat dalam lagu yang sama dibuang: {n0 - len(chunks):,}")

chunks.insert(0, "chunk_id", np.arange(len(chunks)))
chunks["n_songs_same_text"] = chunks.groupby("text_lex")["song_id"].transform("nunique")

Duplikat dalam lagu yang sama dibuang: 213,217


In [4]:
# statistik dan simpan
n = len(chunks)
print("Total chunk              :", f"{n:,}")
print("Chunk per lagu (rerata)  :", round(n / chunks["song_id"].nunique(), 1))
print("Kata per chunk (rerata)  :", round(chunks["text_lex"].str.count(r"\S+").mean(), 1))
print("Chunk ambigu (>1 lagu)   :", f"{(chunks['n_songs_same_text'] > 1).mean():.1%}")
print("Lagu tanpa chunk         :", int(len(df) - chunks['song_id'].nunique()))
print("Perkiraan ukuran embedding fp16 (384-d): %.2f GB" % (n * 384 * 2 / 1e9))

out = f"{OUT_DIR}/chunks_L{CHUNK_LINES}_S{STRIDE}.parquet"
chunks.to_parquet(out, index=False, compression="zstd")
print("Tersimpan:", out, "|", round(os.path.getsize(out) / 1e6, 1), "MB")
display(chunks.head(5))

Total chunk              : 1,843,304
Chunk per lagu (rerata)  : 18.5
Kata per chunk (rerata)  : 20.8
Chunk ambigu (>1 lagu)   : 0.3%
Lagu tanpa chunk         : 0
Perkiraan ukuran embedding fp16 (384-d): 1.42 GB
Tersimpan: /kaggle/working/chunks_L3_S2.parquet | 122.7 MB


,chunk_id,song_id,start_line,end_line,text,text_lex,n_songs_same_text
0,0,0,0,3,"I was once a fool, too\nI know what he's putti...",i was once a fool too i know what hes puttin y...,1
1,1,0,2,5,*So don't tell me what I should do\nI ain't go...,so dont tell me what i should do i aint gon en...,1
2,2,0,4,7,His love has gotten you blind\nWake up and use...,his love has gotten you blind wake up and use ...,1
3,3,0,6,9,"*I let you do your thing\nStop playa, hatin' o...",i let you do your thing stop playa hatin on me...,1
4,4,0,8,11,"Kelly, ain't it a shame?\nThat boy you call yo...",kelly aint it a shame that boy you call your m...,1
